# SSF2 RL — Exploration & Data Collection

This notebook connects to the instrumented SSF2 build, lets you poke at the
observation/action space, drive the character with scripted inputs, and record
trajectories you can later use for supervised learning (behavioral cloning) or
to sanity-check your own RL implementations.

**Prerequisites** (run once, from the repo root):
```bash
cd /Users/cachemiss/Documents/projects/reflash2-fork/reflash2
.venv/bin/pip install -e python          # makes ssf2_rl importable
```

**The game auto-launches.** `env.reset()` (and `BotRunner.start()`) start the
game themselves if it isn't already running — no manual terminal step needed.
Requires `AIR_SDK_HOME` set, or an AIR SDK at `~/Developer/AIRSDK*`.
Game output is logged to `.macos/adl.log`; `ssf2_rl.stop_game()` quits a game
started this way. To start it manually instead:
```bash
AIR_SDK_HOME="$HOME/Developer/AIRSDK_51.3.3" bash tools/macos/run_macos.sh
```

Select the repo's `.venv` as the notebook kernel (Cmd+Shift+P → "Notebook: Select Kernel" → `.venv`).

In [1]:
# Bot layer: ScriptedBot (fixed sequences), ZeroBot (stand still),
# PolicyBot (neural-net ready). BotRunner drives them frame-by-frame.
from ssf2_rl.bots import ZeroBot, ScriptedBot, BotRunner
from ssf2_rl.env import SSF2Env



## 1. Connect & reset

`reset()` restarts the match in-game and takes over player 1, so your inputs
drive Marth while the CPU plays the opponent.

In [3]:
env = SSF2Env()                      # agent_player=1 by default
obs, info = env.reset()

# names = obs_feature_names()
# for i, (n, v) in enumerate(zip(names, obs)):
#     print(f"{i:2d} {n:16s} {v:+.3f}")
# print("\nframe:", info["frame"], "| me:", info["me"]["name"], "| opp:", info["opp"]["name"])

## 2. Scripted control (bot layer)

SSF2 RL uses a Slippi-AI-style bot layer: every control source implements
`Bot.act(state, me_id) -> controls mask`, and a `BotRunner` takes over the
chosen slots and drives them frame-by-frame. Slots left out of the runner
keep their native control source (in-game CPU AI or human).

Available bots: `ScriptedBot` (fixed action sequences), `ZeroBot` (stand
still), `PolicyBot` (any callable obs -> action, neural-net ready).

In [4]:
# --- Both characters stand still -------------------------------------------
# ZeroBot sends mask 0 ("no buttons held") every frame for BOTH slots.
# Note: zero input stops new movement, but residual momentum still decays
# via physics, so x may drift slightly before settling.

env.close()  # free the single bridge connection

runner = BotRunner(bots={1: ZeroBot(), 2: ZeroBot()})
runner.start()
traj = runner.run(frames=300, record=True)  # ~10s at 30 FPS
runner.close()
env.close()
#TODO: currently when the runner finishes running, control is handed back to the in game cpus for both players.
#   I would like to make it so that when this finishes, the bots don't do anything; I don't like how the script finishes
#   and I'm not sure if it's an in game cpu, scripted ai, or human control



In [ ]:
# def char_in(state, pid):
#     return next(c for c in state["chars"] if c["id"] == pid)

# for pid in (1, 2):
#     frames = traj[pid]
#     xs = [char_in(r["state"], pid)["x"] for r in frames]
#     ctrls = [char_in(r["state"], pid)["controls"] for r in frames]
#     zero_frames = sum(1 for c in ctrls if c == 0)
#     print(f"P{pid}: x {xs[0]:+.2f} -> {xs[-1]:+.2f} "
#           f"(drift {xs[-1] - xs[0]:+.2f}) | controls==0 on {zero_frames}/{len(ctrls)} frames")

# env = SSF2Env()
# obs, info = env.reset()

In [5]:
# --- Scripted control via the bot layer ------------------------------------
# The same action sequence as before, now expressed as a ScriptedBot driven
# frame-by-frame by a BotRunner. Player 2 (Samus) is NOT in the bots dict,
# so the native in-game CPU AI keeps controlling her.

env.close()  # the bridge allows only ONE client; free it for the runner

script = [
    ("right", 200),
    ("down_special", 20),
    ("noop", 15),
    ("right", 60),
    ("left", 15),
    ("right", 15),
    ("left", 15),
    ("down_special", 20),
    ("right_jump", 6),
    ("right_attack", 20),
    ("shield", 15),
]

runner = BotRunner(bots={1: ScriptedBot(script, on_end='noop'), 2: ZeroBot()})
runner.start()
traj = runner.run(frames=sum(f for _, f in script), record=True)
runner.close()

# def char_in(state, pid):
#     return next(c for c in state["chars"] if c["id"] == pid)

# me_frames = traj[1]
# xs = [char_in(r["state"], 1)["x"] for r in me_frames]
# ys = [char_in(r["state"], 1)["y"] for r in me_frames]



# Re-create the env so the following cells keep working. The default behavior is for both slots to be CPUs
env = SSF2Env()
obs, info = env.reset()

## 3. Record a random-policy trajectory

Collect `(obs, action, reward, next_obs, done)` tuples with a random policy.
These trajectories are the raw material for behavioral cloning and for
verifying your own value/policy implementations later.

In [ ]:
N_STEPS = 600  # ~20s at 30 FPS

obs, info = env.reset()
traj = {"obs": [], "action": [], "reward": [], "next_obs": [], "done": []}

for t in range(N_STEPS):
    a = env.action_space.sample()
    next_obs, r, term, trunc, info = env.step(a)
    traj["obs"].append(obs)
    traj["action"].append(a)
    traj["reward"].append(r)
    traj["next_obs"].append(next_obs)
    traj["done"].append(term or trunc)
    obs = next_obs
    if term or trunc:
        obs, info = env.reset()

traj = {k: np.asarray(v) for k, v in traj.items()}
print({k: v.shape for k, v in traj.items()})
print("total reward:", round(traj["reward"].sum(), 2), "| mean/step:", round(traj["reward"].mean(), 4))

In [ ]:
# Persist the trajectory for later notebooks / offline analysis.
out = REPO / "notebooks" / "data"
out.mkdir(exist_ok=True)
np.savez_compressed(out / "random_traj.npz", **traj)
print("saved to", out / "random_traj.npz")